# OpenSanctions bundle - London subset

## Set up

In [1]:
import json
import os
import pathlib
import sys
import typing

from icecream import ic
from tqdm import tqdm
import kuzu
import pandas as pd
import watermark

%load_ext watermark

In [2]:
%watermark
%watermark --iversions

Last updated: 2024-08-11T15:34:15.696251-07:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.26.0

Compiler    : Clang 13.0.0 (clang-1300.0.29.30)
OS          : Darwin
Release     : 23.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

pandas   : 2.2.2
sys      : 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
json     : 2.0.9
kuzu     : 0.5.0
watermark: 2.4.3



## Data discovery

In [18]:
dat_dir: pathlib.Path = pathlib.Path("subset")
os_id_set: typing.Set[ str ] = set()

load file: `default.json` for 
> Full sanctions, PEP, crime and associated risk graph

In [19]:
dat_file: pathlib.Path = dat_dir / "default.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7183/7183 [00:00<00:00, 90894.15it/s]


,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,RISKS,ADDRESSES,DATES,COUNTRIES,CONTACTS,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,GENDER
0,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2024-08-07T10:22:04,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BANK SA...","[{'TOPIC': 'sanction'}, {'TOPIC': 'fin.bank'}]","[{'ADDR_FULL': '120 MOORGATE, LONDON EC2M 6TS,...",[{'REGISTRATION_DATE': '1973-08-03'}],"[{'CITIZENSHIP': 'ir'}, {'CITIZENSHIP': 'gb'},...",[{'WEBSITE_ADDRESS': 'https://www.saderat-plc....,"[{'NATIONAL_ID_NUMBER': '01126618'}, {'NATIONA...",[{'SOURCE_URL': 'https://permid.org/1-50008660...,"[{'REL_POINTER_ROLE': 'related-to', 'REL_POINT...",https://www.opensanctions.org/entities/NK-276g...,
1,OPEN_SANCTIONS,NK-2RoX4fFRx9FtQE9osMzwk9,ORGANIZATION,2024-03-06T18:31:06,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'ElEi Ho...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': '61A Asplins Road, London, Unit...",,[{'REGISTRATION_COUNTRY': 'io'}],,"[{'NATIONAL_ID_NUMBER': '13552425'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-2RoX...,
2,OPEN_SANCTIONS,NK-2WqqzAmKXFyrgf7xMDjYMv,PERSON,2024-06-07T08:01:59,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'SHERRI...",[{'TOPIC': 'debarment'}],"[{'ADDR_FULL': 'LONDON, ONTARIO, CANADA, TX'},...",[{'DATE_OF_BIRTH': '1970-03-23'}],[{'CITIZENSHIP': 'us'}],,"[{'OTHER_ID_TYPE': 'OPEN_SANCTIONS', 'OTHER_ID...",,,https://www.opensanctions.org/entities/NK-2Wqq...,


In [20]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,RISKS,ADDRESSES,DATES,COUNTRIES,CONTACTS,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,GENDER
count,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183,7183
freq,7183,1,5456,5177,34,4891,376,5681,5386,6976,1,6104,5598,1,6683
unique,1,7183,3,262,6169,51,5444,1438,263,206,7183,1079,1468,7183,3
top,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2023-03-16T00:00:00,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'Organiz...",[{'TOPIC': 'fin.bank'}],[{'PLACE_OF_BIRTH': 'London'}],,[{'CITIZENSHIP': 'gb'}],,"[{'NATIONAL_ID_NUMBER': '01126618'}, {'NATIONA...",,,https://www.opensanctions.org/entities/NK-276g...,


In [23]:
os_id_set |= set(df.RECORD_ID.values)
len(os_id_set)

7183

load file: `sanctions.json` for 
> Subset: Government-published sanctions lists

In [24]:
dat_file: pathlib.Path = dat_dir / "sanctions.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 306/306 [00:00<00:00, 52559.77it/s]


,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,RISKS,ADDRESSES,COUNTRIES,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,GENDER,DATES,CONTACTS
0,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,ORGANIZATION,2024-05-07T15:53:01,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BANK SA...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': 'PO Box 15175/584, 6th Floor, S...","[{'CITIZENSHIP': 'gb'}, {'CITIZENSHIP': 'ir'}]","[{'NATIONAL_ID_NUMBER': '01126618'}, {'OTHER_I...",[{'SOURCE_URL': 'https://sanctionssearch.ofac....,"[{'REL_POINTER_ROLE': 'related-to', 'REL_POINT...",https://www.opensanctions.org/entities/NK-276g...,,,
1,OPEN_SANCTIONS,NK-2RoX4fFRx9FtQE9osMzwk9,ORGANIZATION,2024-03-06T18:31:06,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'ElEi Ho...",[{'TOPIC': 'sanction'}],[{'ADDR_FULL': 'Сполучене Королівство Великої ...,[{'REGISTRATION_COUNTRY': 'io'}],"[{'NATIONAL_ID_NUMBER': '13552425'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-2RoX...,,,
2,OPEN_SANCTIONS,NK-3KRoVfPHitLoVonaD9y6bE,PERSON,2024-05-09T21:39:15,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'Ashraf...",[{'TOPIC': 'sanction'}],"[{'ADDR_FULL': '1 College Yard, Winchester Ave...","[{'NATIONALITY': 'sd'}, {'NATIONALITY': 'ss'},...","[{'PASSPORT_NUMBER': 'B00018325'}, {'NATIONAL_...",[{'SOURCE_URL': 'https://sanctionssearch.ofac....,[{'REL_POINTER_ROLE': 'Owned or Controlled By'...,https://www.opensanctions.org/entities/NK-3KRo...,M,"[{'DATE_OF_BIRTH': '1957-01-31'}, {'DATE_OF_BI...",


In [25]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,RISKS,ADDRESSES,COUNTRIES,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,GENDER,DATES,CONTACTS
count,306,306,306,306,306,306,306,306,306,306,306,306,306,306,306
freq,306,1,170,109,10,151,149,202,1,192,246,1,263,242,293
unique,1,306,3,34,271,5,145,68,306,115,58,306,3,65,14
top,OPEN_SANCTIONS,NK-276gB7L8sFEhctkuEpZsus,None,2023-11-02T16:38:16,"[{'NAME_TYPE': 'PRIMARY', 'NAME_FULL': 'London'}]",,,[{'CITIZENSHIP': 'gb'}],"[{'NATIONAL_ID_NUMBER': '01126618'}, {'OTHER_I...",,,https://www.opensanctions.org/entities/NK-276g...,,,


In [26]:
os_id_set |= set(df.RECORD_ID.values)
len(os_id_set)

7183

load file: `gleif.json` for 
> Companies that have a legal entity identifier (LEI)

In [27]:
dat_file: pathlib.Path = dat_dir / "gleif.json"

with dat_file.open("r", encoding = "utf-8") as fp:
    df: pd.DataFrame = pd.DataFrame([
        json.loads(line)
        for line in tqdm(fp.readlines(), desc = "read JSON")
    ]).astype(str).fillna("")

df.replace({"nan": ""}, regex = True, inplace = True)
df.head(3)

read JSON: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 63145/63145 [00:00<00:00, 112070.86it/s]


,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,ADDRESSES,DATES,COUNTRIES,IDENTIFIERS,URL,RELATIONSHIPS,RISKS
0,OS_GLEIF,lei-03EINY24LQ6IXW124R72,ORGANIZATION,2024-07-10T08:08:08,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'THE BAR...",[{'ADDR_FULL': 'C/O BARCLAYS PENSION FUNDS TRU...,[{'REGISTRATION_DATE': '2012-09-24'}],[{'REGISTRATION_COUNTRY': 'gb'}],"[{'LEI_NUMBER': '03EINY24LQ6IXW124R72'}, {'OTH...",https://www.opensanctions.org/entities/lei-03E...,,
1,OS_GLEIF,lei-05MQKGBWLLX7RPPDO189,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'THE MAR...","[{'ADDR_FULL': '16 HATFIELDS, LONDON SE1 8DJ, ...",,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'NATIONAL_ID_NUMBER': '03909510'}, {'LEI_NUM...",https://www.opensanctions.org/entities/lei-05M...,,
2,OS_GLEIF,lei-0677F7L8DTFEBS5RWE15,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'CARRING...","[{'ADDR_FULL': '5 CANADA SQUARE, LONDON E14 5A...",,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'NATIONAL_ID_NUMBER': '06345337'}, {'LEI_NUM...",https://www.opensanctions.org/entities/lei-067...,,


In [28]:
df.describe(include = "all").loc[[ "count", "freq", "unique", "top", ]]

,DATA_SOURCE,RECORD_ID,RECORD_TYPE,LAST_CHANGE,NAMES,ADDRESSES,DATES,COUNTRIES,IDENTIFIERS,URL,RELATIONSHIPS,RISKS
count,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145,63145
freq,63145,1,63145,42170,4,440,24802,57099,1,1,49182,60903
unique,1,63145,1,56,63007,35299,10987,99,63145,63145,13962,2
top,OS_GLEIF,lei-03EINY24LQ6IXW124R72,ORGANIZATION,2024-06-12T08:08:21,"[{'NAME_TYPE': 'PRIMARY', 'NAME_ORG': 'BLACKRO...",[{'ADDR_FULL': 'C/O HSBC TRUST COMPANY (UK) LI...,,[{'REGISTRATION_COUNTRY': 'gb'}],"[{'LEI_NUMBER': '03EINY24LQ6IXW124R72'}, {'OTH...",https://www.opensanctions.org/entities/lei-03E...,,


In [29]:
os_id_set |= set(df.RECORD_ID.values)
len(os_id_set)

70317